# 🧾 Verificación reproducible en Kaggle — Agente de Órdenes de Compra SAP

Este notebook **no es un backend aparte**: clona el repositorio y ejecuta, sobre Kaggle, el mismo backend TypeScript (`src/`, herramientas `oc_*`, SAP simulado) para verificarlo de punta a punta:

1. Instala Bun y las dependencias.
2. Corre las 13 pruebas y el `demo.ts` (los 6 casos, **sin modelo de lenguaje**).
3. *(Opcional)* Prueba el agente contra OpenAI usando un Secret de Kaggle.
4. Muestra los artefactos generados (`control.csv`, payloads, evidencia).

> Requiere **Internet On** en la configuración del notebook. Para el paso 3, agrega el Secret `OPENAI_API_KEY` en *Add-ons → Secrets*.

In [ ]:
%%bash
set -e
rm -rf /kaggle/working/agente-oc
git clone --depth 1 https://github.com/jogarzonpa-lang/agente-ordenes-compra-sap.git /kaggle/working/agente-oc
[ -x "$HOME/.bun/bin/bun" ] || curl -fsSL https://bun.sh/install | bash
export PATH=$HOME/.bun/bin:$PATH
cd /kaggle/working/agente-oc && bun install && ls

## 1) Pruebas y demo determinista (sin modelo)

In [ ]:
%%bash
export PATH=$HOME/.bun/bin:$PATH
cd /kaggle/working/agente-oc
bun run test
bun run demo.ts

## 2) Artefactos generados

In [ ]:
import pandas as pd, json, pathlib
base = pathlib.Path("/kaggle/working/agente-oc/out")
display(pd.read_csv(base / "control.csv"))
print(json.dumps(json.loads((base / "sol-004" / "payload.json").read_text()), indent=2, ensure_ascii=False))
print(sorted(p.name for p in (base / "sol-004").iterdir()))

## 3) (Opcional) Prueba real contra el modelo

Consume tokens de tu clave de OpenAI. Ejecuta el prompt de demo del reto sobre `sol-004`, luego confirma, y luego procesa `sol-002` (bloqueada).

In [ ]:
from kaggle_secrets import UserSecretsClient
import os, subprocess
env = {**os.environ,
       "OPENAI_API_KEY": UserSecretsClient().get_secret("OPENAI_API_KEY"),
       "PATH": os.path.expanduser("~/.bun/bin") + ":" + os.environ["PATH"]}
r = subprocess.run(["bun", "run", "probar:modelo"], cwd="/kaggle/working/agente-oc", env=env, capture_output=True, text=True)
print(r.stdout)
print(r.stderr[-1500:])

## 4) Empaquetar las evidencias para descargarlas

In [ ]:
%%bash
cd /kaggle/working/agente-oc && zip -qr /kaggle/working/evidencias.zip out && ls -lh /kaggle/working/evidencias.zip